# Fine-tune ResNet-50 V2 and extract MMSD image features

This notebook follows the paper's Section 3 wording literally: initialize ResNet-50 V2 from ImageNet weights, replace its classifier, fine-tune it on the sarcasm training split, then freeze it and save its 14×14×2048 spatial features. Fine-tuning uses training labels only; validation selects the checkpoint; test labels are not used during training or checkpoint selection.

> Reproduction note: the paper specifies the architecture and feature shape, but not the fine-tuning schedule or checkpoint. The authors' released TensorFlow code consumes precomputed ResNet vectors, and their email permits a pretrained ResNet or another pretrained model. This notebook follows Section 3's fine-tuning description; its optimizer and schedule are explicit choices, not recovered author settings.

## Configure data and run

In [ ]:
import ast, json, os, random, time
from pathlib import Path

import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import timm
from tqdm.auto import tqdm

DATA_ROOT = Path('/kaggle/input/datasets/fruich/mmsd-full/data')
ASSET_ROOT = DATA_ROOT / 'raw/mmsd'
IMAGE_DIR = DATA_ROOT / 'raw/dataset_image'
OUTPUT_DIR = Path('/kaggle/working/features/resnetv2_50_a1h_in1k_finetuned')
MODEL_NAME = 'resnetv2_50.a1h_in1k'
SEED, BATCH_SIZE, MAX_EPOCHS, PATIENCE = 42, 32, 8, 5
BACKBONE_LR, CLASSIFIER_LR = 1e-5, 1e-3
SEQ_LEN = 75
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
assert ASSET_ROOT.is_dir() and IMAGE_DIR.is_dir(), 'Check DATA_ROOT and IMAGE_DIR.'
print('Device:', DEVICE, '| timm:', timm.__version__)

## Load the paper's filtered training and validation examples

In [ ]:
blocked = {'sarcasm', 'sarcastic', 'reposting', '<url>', 'joke', 'humour', 'humor', 'jokes', 'irony', 'ironic', 'exgag'}
def read_split(name):
    rows = [ast.literal_eval(line) for line in (ASSET_ROOT / f'text/{name}.txt').read_text().splitlines() if line.strip()]
    return [row for row in rows if not blocked.intersection(row[1].split())]

train_rows, valid_rows = read_split('train'), read_split('valid2')
print('Train:', len(train_rows), 'Validation:', len(valid_rows))

## Fine-tune the image classifier

The temporary head is an image-only binary sarcasm classifier: its prediction is compared with the sarcasm label of the paired tweet using binary cross-entropy. That error updates both the replacement head and ResNet backbone. The tweet label is only weak supervision for the image, since an image does not always reveal the tweet's sarcasm. After validation selects a checkpoint, the temporary classifier is discarded; the fine-tuned ResNet is frozen and used to extract regional image embeddings for multimodal fusion.

Fine-tuning data: MMSD training images and their tweet labels only. The starting weights are still pretrained on ImageNet; MMSD is the task-specific fine-tuning data. Validation selects the checkpoint, and test labels are excluded. The paper does not report the fine-tuning schedule, so the learning rates below are explicit implementation choices.

In [ ]:
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=1).to(DEVICE)
mean = torch.tensor(model.pretrained_cfg['mean'], dtype=torch.float32).view(3, 1, 1)
std = torch.tensor(model.pretrained_cfg['std'], dtype=torch.float32).view(3, 1, 1)

class ImageRows(Dataset):
    def __init__(self, rows, training):
        self.rows, self.training = rows, training
    def __len__(self): return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(IMAGE_DIR / f'{row[0]}.jpg') as im:
            im = im.convert('RGB').resize((448, 448), Image.Resampling.BICUBIC)
            x = torch.from_numpy(np.asarray(im, dtype=np.float32).copy()).permute(2, 0, 1) / 255.0
        return (x - mean) / std, int(row[-1]), str(row[0])

train_loader = DataLoader(ImageRows(train_rows, True), batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=DEVICE.type == 'cuda')
valid_loader = DataLoader(ImageRows(valid_rows, False), batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=DEVICE.type == 'cuda')
positive = sum(int(row[-1]) for row in train_rows)
pos_weight = torch.tensor([(len(train_rows) - positive) / positive], device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
classifier_params = list(model.get_classifier().parameters())
backbone_params = [p for p in model.parameters() if all(p is not q for q in classifier_params)]
optimizer = torch.optim.Adam([{'params': backbone_params, 'lr': BACKBONE_LR}, {'params': classifier_params, 'lr': CLASSIFIER_LR}])

def f1_at_half(labels, probs):
    pred = probs >= 0.5
    tp = int(((labels == 1) & pred).sum()); fp = int(((labels == 0) & pred).sum()); fn = int(((labels == 1) & ~pred).sum())
    return 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint = OUTPUT_DIR / 'finetuned_resnet50v2.pt'
best_f1, stale, history = -1.0, 0, []
for epoch in range(1, MAX_EPOCHS + 1):
    model.train(); total_loss = 0.0
    for images, labels, _ in tqdm(train_loader, desc=f'Train {epoch}/{MAX_EPOCHS}'):
        images = images.to(DEVICE, non_blocking=True); labels = labels.float().to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(images).squeeze(1), labels)
        if not torch.isfinite(loss): raise FloatingPointError('Non-finite loss')
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); optimizer.step()
        total_loss += loss.item() * len(labels)
    model.eval(); ys, ps = [], []
    with torch.inference_mode():
        for images, labels, _ in valid_loader:
            prob = model(images.to(DEVICE, non_blocking=True)).squeeze(1).sigmoid().cpu().numpy()
            ys.extend(labels.numpy()); ps.extend(prob)
    score = f1_at_half(np.asarray(ys), np.asarray(ps))
    history.append({'epoch': epoch, 'train_loss': total_loss / len(train_rows), 'validation_f1': score})
    print(history[-1])
    if score > best_f1:
        best_f1, stale = score, 0
        torch.save({'model': model.state_dict(), 'epoch': epoch, 'validation_f1': score}, checkpoint)
    else:
        stale += 1
        if stale >= PATIENCE: break
assert checkpoint.is_file()
(OUTPUT_DIR / 'finetune_history.json').write_text(json.dumps(history, indent=2) + '\n')
print('Selected validation F1:', best_f1, '| checkpoint:', checkpoint)

## Feature extraction handoff


This notebook stops after saving `finetuned_resnet50v2.pt`. Run `notebooks/extract_resnet50v2_image_regions.ipynb` three times with `PART=1`, `2`, and `3`; it loads this checkpoint and writes the `[196, 2048]` arrays in resumable parts. The hierarchical training notebook reads those features from the same output directory. No guidance vectors or reconstructed vectors are computed here; those are later fusion-model stages.

## Method notes

The output tensor has the paper/code-compatible shape `[196, 2048]`; the hierarchical model's trainable 2048→1024 projection is still learned during fusion training. The temporary image-only classifier is discarded after fine-tuning; the fine-tuned ResNet produces the embeddings used by the later multimodal classifier. The hierarchical training notebook uses the authors' released top-five attribute predictions and GloVe attribute vectors (`extract/extract_all` and `ExtractWords`); it does not retrain the ResNet-101 COCO attribute predictor.